# Demo — Lineage as a Blast-Radius Map

Lineage answers two separate operational questions: if a dataset breaks or suffers a breach, **what
downstream is affected?** (`trace_downstream` / `simulate_breach`) and for a regulated output,
**what sources feed it?** (`trace_upstream`). This walkthrough reads the Trailhead lineage graph,
traces the shipment-to-carbon pipeline in both directions, and specifies one monitoring signal; the
exercise asks you to answer two independent lineage questions — (1) trace a `customer` PII breach
downstream and (2) verify the CSRD report's upstream provenance — and specify the platform
monitoring table.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog and S3 tables are seeded in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

display(gc.lineage.events())
print("\nMermaid (paste into any Mermaid renderer):\n")
print(gc.lineage.to_mermaid())

## Trace downstream (blast radius) and upstream (provenance)
Use `trace_downstream()` and `simulate_breach()` to see what a defect in `shipment` contaminates,
and `trace_upstream()` to verify which source tables feed the intermediate `order_carbon` table:

In [ ]:
print("Downstream of shipment:", gc.lineage.trace_downstream("shipment"))
print("Upstream of order_carbon:", gc.lineage.trace_upstream("order_carbon"))

breach = gc.lineage.simulate_breach("shipment")
breach

## Place a monitoring signal at the critical junction
Because a defect at `shipment` flows directly into `order_carbon` and then
`quarterly_sustainability` with no gate in between, we place a monitoring signal on that edge:

| Signal | Where (Source dataset/job) | Threshold | Owner | SLA / Regulation |
|---|---|---|---|---|
| `shipment_carbon_range` | `shipment` → `build_order_carbon` | `carbon_kg <= 100` (0 rows above max) | `inventory-fulfillment@trailhead.example` | CSRD Art. 29a carbon accuracy |

**Takeaway / your turn:** in the exercise you'll answer two separate lineage questions — (1) trace a
`customer` PII breach downstream with `trace_downstream("customer")` (confirming it impacts
`customer_360` and `marketing_audience` and does not reach the CSRD report) and (2) trace all
upstream sources of `quarterly_sustainability` with `trace_upstream("quarterly_sustainability")` —
and specify the full monitoring table across the platform.